# Segmentation

**Course:** [Computer Vision](https://ml-viz.vercel.app/courses/computer-vision/02-segmentation)

This notebook implements Dice loss, simulates U-Net skip connection behavior, and compares semantic vs instance segmentation outputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
})

## Semantic vs Instance Segmentation

In [ ]:
def make_segmentation_example(size=128):
    """Create a simple scene with 3 overlapping circles representing objects."""
    y, x = np.ogrid[:size, :size]
    centers = [(35, 40, 25), (70, 60, 22), (55, 90, 20)]  # (cy, cx, r)
    masks = []
    for cy, cx, r in centers:
        masks.append(((y - cy)**2 + (x - cx)**2) < r**2)
    return masks

size = 128
masks = make_segmentation_example(size)

# Semantic: any cell belongs to class 'circle' (foreground=1, background=0)
semantic = np.zeros((size, size), dtype=int)
for m in masks:
    semantic[m] = 1

# Instance: each circle is a different instance
instance = np.zeros((size, size), dtype=int)
for i, m in enumerate(masks):
    instance[m] = i + 1

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Raw image
raw = np.zeros((size, size, 3))
colors = [[0.3, 0.5, 0.9], [0.9, 0.4, 0.3], [0.3, 0.8, 0.5]]
for i, (m, c) in enumerate(zip(masks, colors)):
    raw[m] = c
axes[0].imshow(raw)
axes[0].set_title('Input image', fontsize=11)

# Semantic segmentation
axes[1].imshow(semantic, cmap='Blues', vmin=-0.5, vmax=1.5)
axes[1].set_title('Semantic segmentation\n(all circles = class 1)', fontsize=11)

# Instance segmentation  
cmap = plt.cm.get_cmap('Set1', 4)
axes[2].imshow(instance, cmap=cmap, vmin=0, vmax=3)
axes[2].set_title('Instance segmentation\n(circle 1, 2, 3 distinct)', fontsize=11)

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Dice Loss

Dice loss handles class imbalance better than BCE: it measures overlap directly between prediction and ground truth.

In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    """
    Soft Dice loss.
    pred: (H, W) float in [0, 1] (predicted probabilities)
    target: (H, W) int in {0, 1} (ground truth mask)
    """
    intersection = (pred * target).sum()
    dice_coeff = (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    return 1 - dice_coeff

def bce_loss(pred, target, eps=1e-8):
    return -np.mean(target * np.log(pred + eps) + (1 - target) * np.log(1 - pred + eps))

# Test with tiny foreground object (class imbalance scenario)
gt = np.zeros((64, 64))
gt[30:34, 30:34] = 1  # 4×4 object in 64×64 image — only 0.4% foreground

# Scenario A: model predicts all background (lazy baseline)
pred_all_bg = np.full((64, 64), 0.01)
# Scenario B: model correctly predicts the small object
pred_correct = gt.copy() * 0.9 + 0.05

print("All-background prediction:")
print(f"  BCE  loss: {bce_loss(pred_all_bg, gt):.4f}")
print(f"  Dice loss: {dice_loss(pred_all_bg, gt):.4f}")

print("\nCorrect prediction:")
print(f"  BCE  loss: {bce_loss(pred_correct, gt):.4f}")
print(f"  Dice loss: {dice_loss(pred_correct, gt):.4f}")

print("\nBCE improvement ratio:", round(bce_loss(pred_all_bg, gt) / bce_loss(pred_correct, gt), 2), "x")
print("Dice improvement ratio:", round(dice_loss(pred_all_bg, gt) / dice_loss(pred_correct, gt), 2), "x")

## U-Net skip connections: comparing with and without

In [ ]:
def simulate_encoding(mask, n_stages=4):
    """Simulate downsampling (like U-Net encoder) at multiple scales."""
    scales = [mask]
    current = mask.astype(float)
    for _ in range(n_stages):
        # Downsample by 2x using strided average pooling
        h, w = current.shape
        current = current[:h//2*2, :w//2*2].reshape(h//2, 2, w//2, 2).mean(axis=(1,3))
        scales.append(current)
    return scales

def upsample_bilinear(arr, target_size):
    """Simple nearest-neighbor upsampling."""
    h, w = target_size
    factors = (h // arr.shape[0], w // arr.shape[1])
    return arr.repeat(factors[0], axis=0).repeat(factors[1], axis=1)

# Simulate ground truth mask
size = 64
gt_mask = np.zeros((size, size))
gt_mask[15:25, 20:30] = 1  # thin object
gt_mask[35:40, 40:55] = 1  # another thin object

scales = simulate_encoding(gt_mask)
bottleneck = scales[-1]  # most compressed
original_size = scales[0].shape

# Without skip connections: upsample from bottleneck directly
decoded_no_skip = upsample_bilinear(bottleneck, original_size)

# With skip connections: add back encoder features at each scale
decoded_with_skip = bottleneck.copy()
for enc_feat in reversed(scales[1:-1]):
    decoded_with_skip = upsample_bilinear(decoded_with_skip, enc_feat.shape)
    decoded_with_skip = np.clip(decoded_with_skip + 0.7 * enc_feat, 0, 1)
decoded_with_skip = upsample_bilinear(decoded_with_skip, original_size)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(gt_mask, cmap='viridis', vmin=0, vmax=1)
axes[0].set_title('Ground truth mask', fontsize=11)
axes[1].imshow(decoded_no_skip, cmap='viridis', vmin=0, vmax=1)
axes[1].set_title('Decoded without skip connections\n(blurry, lost fine details)', fontsize=11)
axes[2].imshow(decoded_with_skip, cmap='viridis', vmin=0, vmax=1)
axes[2].set_title('Decoded with skip connections\n(sharper object boundaries)', fontsize=11)
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## ✏️ Your turn

### Exercise 1: Implement Dice coefficient

Implement the Dice coefficient (not loss) for binary masks.

In [ ]:
def dice_coefficient(pred_mask, true_mask, smooth=1e-6):
    """
    Compute Dice coefficient = 2|A∩B| / (|A| + |B|).
    
    Args:
        pred_mask: np.ndarray of bool or {0,1}
        true_mask: np.ndarray of bool or {0,1}
        smooth: float to avoid division by zero
    Returns:
        float: Dice coefficient in [0, 1], 1 = perfect overlap
    """
    # TODO(you): compute 2 × intersection / (|pred| + |true|)
    pass


# Test
a = np.array([[1, 1, 0], [1, 0, 0], [0, 0, 0]])
b = np.array([[1, 1, 0], [1, 0, 0], [0, 0, 0]])
print(f"Identical masks: {dice_coefficient(a, b):.4f} (should be ~1.0)")
print(f"No overlap:      {dice_coefficient(a, 1-a):.4f} (should be ~0.0)")

In [ ]:
a = np.array([[1,1,0],[1,0,0],[0,0,0]])
b_half = np.array([[1,0,0],[0,0,0],[0,0,0]])
d_identical = dice_coefficient(a, a)
d_nooverlap = dice_coefficient(a, 1-a)
d_half = dice_coefficient(a, b_half)
assert d_identical is not None, "Should return a value"
assert abs(d_identical - 1.0) < 1e-3, f"Identical masks should give Dice≈1, got {d_identical}"
assert d_nooverlap < 0.01, f"No overlap should give Dice≈0, got {d_nooverlap}"
assert 0.4 < d_half < 0.8, f"Half overlap expected ~0.67, got {d_half}"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def dice_coefficient(pred_mask, true_mask, smooth=1e-6):
    intersection = (pred_mask * true_mask).sum()
    return (2 * intersection + smooth) / (pred_mask.sum() + true_mask.sum() + smooth)
```
</details>

### Exercise 2: Compute mIoU for semantic segmentation

mIoU = mean IoU across all classes. IoU per class = TP / (TP + FP + FN).

In [ ]:
def mean_iou(pred_labels, true_labels, num_classes):
    """
    Compute mean IoU for semantic segmentation.
    
    Args:
        pred_labels: np.ndarray (H, W) int — predicted class per pixel
        true_labels: np.ndarray (H, W) int — ground truth class per pixel
        num_classes: int
    Returns:
        float: mean IoU across all classes
    """
    ious = []
    for c in range(num_classes):
        # TODO(you): compute IoU for class c
        # TP = pixels predicted as c AND truly c
        # FP = pixels predicted as c but truly NOT c
        # FN = pixels truly c but predicted NOT c
        # IoU_c = TP / (TP + FP + FN) — skip if no pixels of class c exist
        pass
    return np.mean(ious) if ious else 0.0


pred = np.array([[0, 0, 1, 1],
                 [0, 1, 1, 2],
                 [0, 0, 2, 2]])
true = np.array([[0, 0, 1, 1],
                 [0, 0, 1, 2],
                 [0, 0, 2, 2]])
miou = mean_iou(pred, true, num_classes=3)
print(f"mIoU: {miou:.3f}")

In [ ]:
# Perfect prediction should give mIoU=1.0
assert abs(mean_iou(true, true, 3) - 1.0) < 1e-4, "Perfect prediction should give mIoU=1"
# Partial prediction should be between 0 and 1
miou_partial = mean_iou(pred, true, 3)
assert 0 < miou_partial < 1, f"mIoU should be between 0 and 1, got {miou_partial}"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def mean_iou(pred_labels, true_labels, num_classes):
    ious = []
    for c in range(num_classes):
        tp = ((pred_labels == c) & (true_labels == c)).sum()
        fp = ((pred_labels == c) & (true_labels != c)).sum()
        fn = ((pred_labels != c) & (true_labels == c)).sum()
        if tp + fp + fn > 0:
            ious.append(tp / (tp + fp + fn))
    return np.mean(ious) if ious else 0.0
```
</details>